# 02a - Classical baseline: word features + linear classifier

Runs on a plain CPU in about a minute, so it needs no GPU. Run notebook 01 first (it creates the train / val / test files).

Why this notebook exists: a trained model needs something simple to be compared against. Here every word is described by
its own shape and its neighbours (capital letter, digits, suffix, position in the resume, the words either side) and a
linear SVM predicts its tag. DistilBERT (notebook 02) has to beat this number to justify its cost.

Rules we follow: tune only on the **validation** set, and look at the **test** set once.

In [1]:

import os, sys, json, glob
os.chdir(r"C:\Users\gupta\OneDrive\Desktop\Interv")
sys.path.insert(0, "ml/src/resume_ner")
import data
OUT = r"C:\Users\gupta\OneDrive\Desktop\Interv\ml\data\processed\local_run"
os.makedirs(OUT, exist_ok=True)


In [2]:
import time, ast, joblib, numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_recall_fscore_support
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2
import baseline

load = lambda n: data.load_jsonl(f"{OUT}/{n}_resumes.jsonl")
train, val, test = load("train"), load("val"), load("test")
print("resumes train/val/test:", len(train), len(val), len(test))

def feats_and_tags(rows):
    X = [f for ex in rows for f in baseline.featurize(ex["tokens"])]
    y = [t for ex in rows for t in ex["tags"]]
    return X, y

Xtr_d, ytr = feats_and_tags(train); Xva_d, yva = feats_and_tags(val); Xte_d, yte = feats_and_tags(test)
vec = DictVectorizer()
Xtr = vec.fit_transform(Xtr_d)       # the vocabulary is learnt from TRAIN only
Xva, Xte = vec.transform(Xva_d), vec.transform(Xte_d)
print("training words:", Xtr.shape[0], "| distinct features:", Xtr.shape[1])

resumes train/val/test: 159 20 20


training words: 84306 | distinct features: 78360


## Tune on the validation set

In [3]:
kw = dict(mode="strict", scheme=IOB2)

def to_sequences(rows, flat_pred):
    out, i = [], 0
    for ex in rows:
        n = len(ex["tokens"])
        out.append(baseline.fix_bio(list(flat_pred[i:i + n]))); i += n
    return out

gold_val = [ex["tags"] for ex in val]
grid, best = [], None
for C in [0.05, 0.2, 1.0]:
    for cw in [None, "balanced"]:
        t = time.perf_counter()
        clf = LinearSVC(C=C, class_weight=cw, max_iter=3000).fit(Xtr, ytr)
        f1 = f1_score(gold_val, to_sequences(val, clf.predict(Xva)), **kw)
        grid.append({"C": C, "class_weight": cw, "val_entity_F1": round(f1, 3), "fit_seconds": round(time.perf_counter() - t, 1)})
        if best is None or f1 > best[0]: best = (f1, C, cw)
display(pd.DataFrame(grid))
print("best on validation:", best)

,C,class_weight,val_entity_F1,fit_seconds
0,0.05,NaN,0.376,13.5
1,0.05,balanced,0.318,16.1
2,0.20,NaN,0.346,15.9
3,0.20,balanced,0.318,33.6
4,1.00,NaN,0.304,24.2
5,1.00,balanced,0.305,95.4


best on validation: (np.float64(0.37634408602150543), 0.05, None)


## Final model, evaluated once on the test set

In [4]:
_, C, cw = best
t = time.perf_counter()
clf = LinearSVC(C=C, class_weight=cw, max_iter=3000).fit(Xtr, ytr)
train_seconds = time.perf_counter() - t

gold_test = [ex["tags"] for ex in test]
pred_test = to_sequences(test, clf.predict(Xte))
print(classification_report(gold_test, pred_test, digits=3, **kw))
overall = {"precision": precision_score(gold_test, pred_test, **kw), "recall": recall_score(gold_test, pred_test, **kw),
           "f1": f1_score(gold_test, pred_test, **kw)}
print("overall (strict entity match):", {k: round(v, 3) for k, v in overall.items()})

                     precision    recall  f1-score   support

       COLLEGE_NAME      0.200     0.194     0.197        31
COMPANIES_WORKED_AT      0.419     0.316     0.360        57
             DEGREE      0.458     0.458     0.458        24
        DESIGNATION      0.419     0.367     0.391        49
      EMAIL_ADDRESS      0.800     0.941     0.865        17
    GRADUATION_YEAR      0.000     0.000     0.000        20
           LOCATION      0.667     0.500     0.571        32
               NAME      1.000     0.952     0.976        21
             SKILLS      0.054     0.130     0.076        23
YEARS_OF_EXPERIENCE      0.000     0.000     0.000         4

          micro avg      0.414     0.388     0.401       278
          macro avg      0.402     0.386     0.389       278
       weighted avg      0.427     0.388     0.403       278

overall (strict entity match): {'precision': np.float64(0.414), 'recall': np.float64(0.388), 'f1': np.float64(0.401)}


C:\Users\gupta\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## Compared with the dictionary parser currently in the app

`backend/resume_service.py` finds skills by looking words up in a fixed list. Both systems are scored on **SKILLS**
word by word (the dataset labels whole skills paragraphs, so strict span matching would be unfair to the dictionary).

In [5]:
tree = ast.parse(open("backend/resume_service.py").read())
vocab = {v.lower() for v in next(ast.literal_eval(n.value) for n in tree.body
         if isinstance(n, ast.Assign) and getattr(n.targets[0], "id", "") == "SKILL_VOCAB")}

def prf(g, p):
    a, b, c, _ = precision_recall_fscore_support(g, p, average="binary", zero_division=0)
    return {"precision": round(a, 3), "recall": round(b, 3), "f1": round(c, 3)}

gold_sk = [t.endswith("SKILLS") for tags in gold_test for t in tags]
model_sk = [t.endswith("SKILLS") for tags in pred_test for t in tags]
dict_sk = [w.lower().strip(",.;:()") in vocab for ex in test for w in ex["tokens"]]
cmp = pd.DataFrame([{"system": "Dictionary lookup (current app)", **prf(gold_sk, dict_sk)},
                    {"system": "Word features + linear SVM", **prf(gold_sk, model_sk)}])
display(cmp)

,system,precision,recall,f1
0,Dictionary lookup (current app),0.417,0.029,0.054
1,Word features + linear SVM,0.875,0.201,0.327


## Where it goes wrong

In [6]:
conf = Counter()
for g, p in zip(gold_test, pred_test):
    for a, b in zip(g, p):
        if a != b: conf[(a, b)] += 1
print("Most common mistakes (gold -> predicted):")
for (a, b), n in conf.most_common(10): print(f"  {n:5d}  {a:24s} -> {b}")

Most common mistakes (gold -> predicted):
    824  I-SKILLS                 -> O
     35  I-DESIGNATION            -> O
     34  I-COLLEGE_NAME           -> O
     31  B-COMPANIES_WORKED_AT    -> O
     24  I-SKILLS                 -> B-SKILLS
     20  B-DESIGNATION            -> O
     20  B-GRADUATION_YEAR        -> O
     19  I-COMPANIES_WORKED_AT    -> O
     18  B-COLLEGE_NAME           -> O
     17  O                        -> B-SKILLS


## Efficiency, and save the model

In [7]:
sample = test[0]["tokens"][:400]
t = time.perf_counter()
for _ in range(20):
    baseline.fix_bio(list(clf.predict(vec.transform(baseline.featurize(sample)))))
ms = round((time.perf_counter() - t) / 20 * 1000, 1)

path = f"{OUT}/baseline_model.joblib"
joblib.dump({"vec": vec, "clf": clf}, path)
size_mb = round(os.path.getsize(path) / 1e6, 1)
eff = {"train_seconds": round(train_seconds, 1), "ms_per_400_words_CPU": ms, "model_size_MB": size_mb}
print(eff)

report = {"model": "DictVectorizer + LinearSVC", "best_C": C, "class_weight": cw, "test_resumes": len(test),
          "overall_strict_entity_level": overall, "skills_word_level_comparison": cmp.to_dict("records"),
          "efficiency": eff, "validation_grid": grid}
json.dump(report, open(f"{OUT}/baseline_report.json", "w"), indent=1)
print("saved", path)

{'train_seconds': 11.3, 'ms_per_400_words_CPU': 15.5, 'model_size_MB': 15.1}
saved C:\Users\gupta\OneDrive\Desktop\Interv\ml\data\processed\local_run/baseline_model.joblib


**Write down for the report:** the overall F1, which labels are strong or weak and why (see the per-label table above),
how much better it is than the dictionary parser, and that this is the number DistilBERT has to beat.
Caveat to state: the test set has only 20 resumes, so a label like YEARS_OF_EXPERIENCE (a handful of examples) is very noisy.